# Tracking Metrics

Tracking metrics are recorded on every trial, but they are not optimization
objectives. This is useful for values like:

- cost
- experiment duration
- reagent consumption
- energy
- waste

The optimizer still searches using the objective, but the extra metrics are
preserved for analysis and later constraints.


In [ ]:
import warnings
import matplotlib.pyplot as plt
import pandas as pd
from ax.service.ax_client import AxClient, ObjectiveProperties

warnings.filterwarnings("ignore", message=".*RandomModelBridge does not support prediction.*")

def evaluate_process(temp: float, catalyst: float) -> dict[str, tuple[float, float]]:
    yield_ = 0.85 - (temp - 70.0) ** 2 / 900.0 - (catalyst - 0.35) ** 2 / 0.03
    energy_kwh = 1.20 + 0.015 * temp + 0.60 * catalyst
    run_time_min = 50.0 - 0.20 * temp + 15.0 * catalyst
    return {
        "yield": (yield_, 0.0),
        "energy_kwh": (energy_kwh, 0.0),
        "run_time_min": (run_time_min, 0.0),
    }

client = AxClient(
    random_seed=0,
    enforce_sequential_optimization=False,
    verbose_logging=False,
)
client.create_experiment(
    parameters=[
        {"name": "temp", "type": "range", "bounds": [40.0, 90.0], "value_type": "float"},
        {"name": "catalyst", "type": "range", "bounds": [0.10, 0.60], "value_type": "float"},
    ],
    objectives={"yield": ObjectiveProperties(minimize=False)},
    tracking_metric_names=["energy_kwh", "run_time_min"],
)

for _ in range(5):
    params, trial_index = client.get_next_trial()
    client.complete_trial(trial_index, evaluate_process(**params))

print("Tracking metrics defined on experiment:")
print(sorted(metric.name for metric in client.experiment.tracking_metrics.values()))

df = client.get_trials_data_frame()[["trial_index", "temp", "catalyst", "yield", "energy_kwh", "run_time_min"]].copy()
df


In [ ]:
client.add_tracking_metrics(["waste_g"])

params, extra_trial_index = client.get_next_trial()
extra_data = evaluate_process(**params)
extra_data["waste_g"] = (2.50 + 3.00 * params["catalyst"], 0.0)
client.complete_trial(extra_trial_index, extra_data)

print("Tracking metrics after adding waste_g:")
print(sorted(metric.name for metric in client.experiment.tracking_metrics.values()))

df = client.get_trials_data_frame()[["trial_index", "yield", "energy_kwh", "run_time_min", "waste_g"]].copy()
df.tail()


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].plot(df["trial_index"], df["yield"], marker="o")
axes[0].set_title("Objective")
axes[0].set_xlabel("trial")
axes[0].set_ylabel("yield")

axes[1].plot(df["trial_index"], df["energy_kwh"], marker="o", color="tab:orange")
axes[1].set_title("Tracking: energy")
axes[1].set_xlabel("trial")
axes[1].set_ylabel("kWh")

axes[2].plot(df["trial_index"], df["run_time_min"], marker="o", color="tab:green")
axes[2].set_title("Tracking: runtime")
axes[2].set_xlabel("trial")
axes[2].set_ylabel("minutes")

plt.tight_layout()
plt.show()


## Why This Matters For Your Platform

If you expose tracking metrics in CEID, users can optimize one thing and still
retain a full operational record for:

- cost dashboards
- sustainability metrics
- post-hoc constraints
- audit trail exports
